In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

PATH = "/kaggle/input/competitions/ctrl-is-hers-final/"
train = pd.read_csv(PATH + "train.csv").sort_values("id")
test = pd.read_csv(PATH + "test.csv")
sample = pd.read_csv(PATH + "sample_submission.csv")

y = train["target"]
X = train.drop(columns=["target", "id", "publish_date"])
X_test = test.drop(columns=["id", "publish_date"])

log_cols = ["n_tokens_content", "num_hrefs", "num_self_hrefs", "num_imgs", "num_videos",
            "kw_max_min", "kw_avg_min", "kw_min_max", "kw_max_max", "kw_avg_max",
            "kw_max_avg", "kw_avg_avg", "self_reference_min_shares",
            "self_reference_max_shares", "self_reference_avg_shares"]
for c in log_cols:
    X[c] = np.log1p(X[c].clip(lower=0))
    X_test[c] = np.log1p(X_test[c].clip(lower=0))

cat_cols = ["weekday", "channel"]
num_cols = [c for c in X.columns if c not in cat_cols]
prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

cut = int(len(X) * 0.8)
X_tr, X_va = X.iloc[:cut], X.iloc[cut:]
y_tr, y_va = y.iloc[:cut], y.iloc[cut:]

models = {
    "Logistic": LogisticRegression(max_iter=2000),
    "Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, random_state=42, verbose=-1),
}

for name, m in models.items():
    pipe = Pipeline([("prep", prep), ("model", m)])
    pipe.fit(X_tr, y_tr)
    print(name, roc_auc_score(y_va, pipe.predict_proba(X_va)[:, 1]))

final = Pipeline([("prep", prep), ("model", models["LightGBM"])])
final.fit(X, y)
sub = pd.DataFrame({"id": test["id"], "target": final.predict_proba(X_test)[:, 1]})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.shape)

Logistic 0.7377635153939641
Forest 0.7422802928827569


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM 0.7518496383014082
(7929, 2)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
